In [0]:
%run "../../utils/00_config"

In [0]:
from pyspark.sql import functions as F

# Load source tables
price_history = spark.table(
    get_gold_table("marts", "mart_item_price_history")
)

dim_execution = spark.table(
    get_table("intermediate", "dim_execution")
)

print(f"Price history records: {price_history.count():,}")
print(f"Execution records: {dim_execution.count():,}")

In [0]:
# Prepare execution metadata
execution_metadata = (
    dim_execution
    .select(
        "execution_id",
        F.col("started_at").alias("execution_started_at"),
        F.col("finished_at").alias("execution_finished_at"),
        "execution_duration_seconds",
        F.col("status").alias("execution_status"),
        F.col("response_count").alias("execution_response_count"),
        F.col("item_count").alias("execution_item_count"),
        F.col("total_orders").alias("execution_total_orders")
    )
)

In [0]:
# Validate execution metadata grain
duplicate_executions = (
    execution_metadata
    .groupBy("execution_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_executions > 0:
    raise ValueError(
        f"Execution metadata contains {duplicate_executions} duplicated execution IDs"
    )

# Enrich offer history with execution metadata
rel_item_price_history = (
    price_history
    .join(
        execution_metadata,
        on="execution_id",
        how="left"
    )
)

# Validate record count preservation
source_count = price_history.count()
reporting_count = rel_item_price_history.count()

if source_count != reporting_count:
    raise ValueError(
        f"Record count mismatch: source={source_count}, reporting={reporting_count}"
    )

# Validate execution metadata coverage
missing_executions = (
    rel_item_price_history
    .select("execution_id")
    .distinct()
    .join(
        execution_metadata.select("execution_id"),
        on="execution_id",
        how="left_anti"
    )
    .count()
)

if missing_executions > 0:
    raise ValueError(
        f"Found {missing_executions} executions without metadata"
    )

print(f"Reporting records: {reporting_count:,}")

In [0]:
'''display(
    rel_item_price_history
    .orderBy(
        "item_slug",
        "observed_at",
        "order_type",
        "platinum"
    )
    .limit(100)
)'''

In [0]:
# Validate offer history grain
display(
    rel_item_price_history.agg(
        F.count("*").alias("total_records"),
        F.countDistinct("execution_id", "order_id").alias("unique_execution_orders"),
        F.countDistinct("execution_id").alias("unique_executions"),
        F.countDistinct("item_slug").alias("unique_items"),
        F.sum(F.when(F.col("order_type") == "buy", 1).otherwise(0)).alias("buy_offers"),
        F.sum(F.when(F.col("order_type") == "sell", 1).otherwise(0)).alias("sell_offers")
    )
)

# Check duplicated offers
duplicates = (
    rel_item_price_history
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Check missing execution metadata
missing_metadata = (
    rel_item_price_history
    .filter(
        F.col("execution_started_at").isNull()
        | F.col("execution_status").isNull()
    )
    .count()
)

# Check missing offer identifiers
missing_order_ids = (
    rel_item_price_history
    .filter(
        F.col("execution_id").isNull()
        | F.col("order_id").isNull()
    )
    .count()
)

print(f"Duplicated execution orders: {duplicates:,}")
print(f"Records with missing execution metadata: {missing_metadata:,}")
print(f"Records with missing identifiers: {missing_order_ids:,}")

if duplicates > 0:
    raise ValueError("Duplicate execution-order combinations detected")

if missing_metadata > 0:
    raise ValueError("Missing execution metadata detected")

if missing_order_ids > 0:
    raise ValueError("Missing execution or order IDs detected")

In [0]:
# Load reporting dependencies explicitly
market_summary = spark.table(
    get_gold_table("marts", "mart_item_price_summary")
)

rel_market_overview = spark.table(
    get_gold_table("reporting", "rel_market_overview")
)

# Validate order type distribution across source and reporting tables
print("MART ITEM PRICE HISTORY")

display(
    price_history
    .groupBy("order_type")
    .count()
    .orderBy("order_type")
)

print("REPORTING ITEM PRICE HISTORY")

display(
    rel_item_price_history
    .groupBy("order_type")
    .count()
    .orderBy("order_type")
)

# Compare source and reporting offer counts
print("SOURCE: MART ITEM PRICE SUMMARY")

display(
    market_summary
    .groupBy("order_type")
    .agg(
        F.sum("total_offers").alias("total_offers")
    )
    .orderBy("order_type")
)

print("REPORTING: MARKET OVERVIEW")

display(
    rel_market_overview
    .agg(
        F.sum("buy_total_offers").alias("buy_offers"),
        F.sum("sell_total_offers").alias("sell_offers"),
        F.sum("total_market_offers").alias("total_offers")
    )
)

In [0]:
from delta.tables import DeltaTable

TARGET_TABLE = get_gold_table(
    "reporting",
    "rel_item_price_history"
)

merge_condition = """
    target.execution_id = source.execution_id
    AND target.order_id = source.order_id
"""

if spark.catalog.tableExists(TARGET_TABLE):
    (
        DeltaTable.forName(spark, TARGET_TABLE)
        .alias("target")
        .merge(
            rel_item_price_history.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        rel_item_price_history
        .write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(TARGET_TABLE)
    )

print(f"Table successfully materialized: {TARGET_TABLE}")

In [0]:
# Validate materialized reporting table
saved_history = spark.table(TARGET_TABLE)

expected_count = rel_item_price_history.count()
actual_count = saved_history.count()

duplicate_count = (
    saved_history
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if expected_count != actual_count:
    raise ValueError(
        f"Record count mismatch: expected {expected_count}, found {actual_count}"
    )

if duplicate_count > 0:
    raise ValueError(
        f"Found {duplicate_count} duplicated execution-order combinations"
    )

print(f"Saved records: {actual_count:,}")
print(f"Duplicated execution-order combinations: {duplicate_count:,}")

'''display(
    saved_history
    .orderBy(
        "item_slug",
        "observed_at",
        "order_type",
        "platinum"
    )
    .limit(100)
)'''